In [1]:
import pandas as pd
from datetime import datetime

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Feature Data

In [3]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
all_data_df={}
all_target_df={}
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_data_df[season]=pd.read_csv(f'{season_path}/all_data_df.csv')
    all_data_df[season]['date']=all_data_df[season]['date'].apply(lambda s: datetime.strptime(s, '%B-%d-%Y'))
    all_data_df[season]=all_data_df[season].sort_values('date').reset_index(drop=True)

    all_target_df[season]=pd.read_csv(f'{season_path}/all_target_df.csv')
    all_target_df[season]['date']=all_target_df[season]['date'].apply(lambda s: datetime.strptime(s, '%B-%d-%Y'))
    all_target_df[season]=all_target_df[season].sort_values('date').reset_index(drop=True)

In [4]:
for season in seasons:
    all_teams=all_data_df[season]['home'].unique().tolist()
    all_team_date_dff=[]
    team_temp_counter={}
    for team in all_teams:
        team_temp_counter[team]=0
        team_data_df=all_data_df[season][(all_data_df[season]['home']==team)|(all_data_df[season]['away']==team)]
        date_diff=team_data_df['date'].iloc[1:].reset_index(drop=True)-team_data_df['date'].iloc[:-1].reset_index(drop=True)
        date_diff=date_diff.apply(lambda x: x.days)
        date_diff.index=date_diff.index+1
        date_diff[0]=-1
        date_diff=date_diff.sort_index()
        all_team_date_dff.append(date_diff)
    all_team_date_dff=pd.concat(all_team_date_dff, axis=1)
    all_team_date_dff.columns=all_teams

    home_day_diff, away_day_diff=[], []
    for _, row in all_data_df[season].iterrows():
        home, away=row['home'], row['away']
        home_day_diff.append(all_team_date_dff.loc[team_temp_counter[home], home])
        team_temp_counter[home]+=1
        away_day_diff.append(all_team_date_dff.loc[team_temp_counter[away], away])
        team_temp_counter[away]+=1
    all_data_df[season]['home_day_diff']=home_day_diff
    all_data_df[season]['away_day_diff']=away_day_diff
    #all_data_df[season]['date_since_last_match']=date_diff
    #all_data_df[season]=all_data_df[season].drop(columns=['date'])
    

In [5]:
all_data_df['2023-24'].head(10)

,home_performance_pk,home_performance_pkatt,home_performance_crdr,home_performance_touches,home_performance_tkl,home_performance_int,home_performance_blocks,home_expected_xg,home_expected_npxg,home_expected_xag,...,away_performance_og,away_performance_recov,away_aerial_duels_won,away_aerial_duels_lost,away_aerial_duels_won%,home,away,date,home_day_diff,away_day_diff
0,0,0,1,496,12,7,12,0.3,0.3,0.3,...,0,54,13,13,50.0,Burnley,Manchester City,2023-08-11,-1,-1
1,0,0,0,693,17,7,17,1.4,1.4,0.3,...,0,52,29,16,64.4,Bournemouth,West Ham,2023-08-12,-1,-1
2,0,0,0,646,16,7,12,3.4,3.4,3.3,...,0,44,6,5,54.5,Newcastle Utd,Aston Villa,2023-08-12,-1,-1
3,0,0,0,902,19,7,6,0.8,0.8,0.6,...,0,34,20,12,62.5,Arsenal,Nott'ham Forest,2023-08-12,-1,-1
4,0,0,0,441,22,11,23,0.5,0.5,0.4,...,0,55,31,13,70.5,Sheffield Utd,Crystal Palace,2023-08-12,-1,-1
5,0,0,0,531,13,13,13,2.7,2.7,2.4,...,0,43,14,9,60.9,Everton,Fulham,2023-08-12,-1,-1
6,1,1,0,769,13,4,8,4.0,3.2,2.9,...,0,41,21,12,63.6,Brighton,Luton Town,2023-08-12,-1,-1
7,1,1,0,464,17,8,15,2.2,1.4,1.3,...,0,50,11,15,42.3,Brentford,Tottenham,2023-08-13,-1,-1
8,0,0,0,862,13,7,14,1.4,1.4,1.3,...,0,62,6,11,35.3,Chelsea,Liverpool,2023-08-13,-1,-1
9,0,0,0,643,18,8,21,2.2,2.2,2.2,...,0,69,17,7,70.8,Manchester Utd,Wolves,2023-08-14,-1,-1


In [6]:
all_data_df['2023-24']['home']+'-'+all_data_df['2023-24']['away']+'-'+all_data_df['2023-24']['date'].apply(lambda x: datetime.strftime(x, "%m/%d/%Y"))

0           Burnley-Manchester City-08/11/2023
1              Bournemouth-West Ham-08/12/2023
2         Newcastle Utd-Aston Villa-08/12/2023
3           Arsenal-Nott'ham Forest-08/12/2023
4      Sheffield Utd-Crystal Palace-08/12/2023
                        ...                   
375                Liverpool-Wolves-05/19/2024
376         Brighton-Manchester Utd-05/19/2024
377             Chelsea-Bournemouth-05/19/2024
378               Luton Town-Fulham-05/19/2024
379         Burnley-Nott'ham Forest-05/19/2024
Length: 380, dtype: object

In [7]:
feature_df=pd.concat((all_data_df.values()), axis=0)
feature_df=feature_df.sort_values(['date','home','away'])
feature_df.index=feature_df['home']+'-'+feature_df['away']+'-'+feature_df['date'].apply(lambda x: datetime.strftime(x, "%m/%d/%Y"))

In [8]:
unique_teams=set(feature_df['home'].tolist()+feature_df['away'].tolist())

In [9]:
team_matches=dict((team, []) for team in unique_teams)
for ind, row in feature_df.iterrows():
    home_team, away_team = row['home'], row['away']
    team_matches[home_team].append(ind)
    team_matches[away_team].append(ind)

# Construct historical feature

In [10]:
history=5

In [11]:
feature_df

,home_performance_pk,home_performance_pkatt,home_performance_crdr,home_performance_touches,home_performance_tkl,home_performance_int,home_performance_blocks,home_expected_xg,home_expected_npxg,home_expected_xag,...,away_performance_og,away_performance_recov,away_aerial_duels_won,away_aerial_duels_lost,away_aerial_duels_won%,home,away,date,home_day_diff,away_day_diff
Liverpool-Norwich City-08/09/2019,0,0,0,627,21,14,11,1.8,1.8,1.6,...,1,37,7,15,31.8,Liverpool,Norwich City,2019-08-09,-1,-1
Bournemouth-Sheffield Utd-08/10/2019,0,0,0,633,18,13,6,1.8,1.8,1.1,...,0,53,20,13,60.6,Bournemouth,Sheffield Utd,2019-08-10,-1,-1
Burnley-Southampton-08/10/2019,0,0,0,507,21,14,14,0.9,0.9,0.6,...,0,53,22,22,50.0,Burnley,Southampton,2019-08-10,-1,-1
Crystal Palace-Everton-08/10/2019,0,0,0,427,21,10,16,0.9,0.9,0.9,...,0,58,18,30,37.5,Crystal Palace,Everton,2019-08-10,-1,-1
Tottenham-Aston Villa-08/10/2019,0,0,0,749,17,5,9,2.5,2.5,1.5,...,0,39,12,10,54.5,Tottenham,Aston Villa,2019-08-10,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Nott'ham Forest-Fulham-09/28/2024,0,0,0,494,8,9,13,0.8,0.8,0.6,...,0,29,7,18,28.0,Nott'ham Forest,Fulham,2024-09-28,6,7
Wolves-Liverpool-09/28/2024,0,0,0,561,14,9,7,0.6,0.6,0.2,...,0,45,10,10,50.0,Wolves,Liverpool,2024-09-28,7,7
Ipswich Town-Aston Villa-09/29/2024,0,0,0,527,18,7,5,1.2,1.2,1.1,...,0,33,7,8,46.7,Ipswich Town,Aston Villa,2024-09-29,8,8
Manchester Utd-Tottenham-09/29/2024,0,0,1,554,27,17,14,1.0,1.0,0.8,...,0,46,8,6,57.1,Manchester Utd,Tottenham,2024-09-29,8,8


In [12]:
historical_feature=[]
hist_ind=[]
for ind, row in feature_df.iterrows():
    match_feature=[]
    home_team, away_team = row['home'], row['away']
    home_match_ix=team_matches[home_team].index(ind)
    away_match_ix=team_matches[away_team].index(ind)
    if home_match_ix<history or away_match_ix<history:
        continue
    for i in range(history):
        match_feature.append(feature_df.loc[team_matches[home_team][home_match_ix-1-i]].drop(['home', 'away', 'date']).rename(lambda x: f'home_{x}_{-i-1}'))
        match_feature.append(feature_df.loc[team_matches[away_team][away_match_ix-1-i]].drop(['home', 'away', 'date']).rename(lambda x: f'away_{x}_{-i-1}'))
    match_feature=pd.concat(match_feature)
    historical_feature.append(match_feature)
    hist_ind.append(ind)
historical_feature=pd.DataFrame(historical_feature)
historical_feature.index=hist_ind

In [13]:
historical_feature.head()

,home_home_performance_pk_-1,home_home_performance_pkatt_-1,home_home_performance_crdr_-1,home_home_performance_touches_-1,home_home_performance_tkl_-1,home_home_performance_int_-1,home_home_performance_blocks_-1,home_home_expected_xg_-1,home_home_expected_npxg_-1,home_home_expected_xag_-1,...,away_away_performance_tklw_-5,away_away_performance_pkwon_-5,away_away_performance_pkcon_-5,away_away_performance_og_-5,away_away_performance_recov_-5,away_away_aerial_duels_won_-5,away_away_aerial_duels_lost_-5,away_away_aerial_duels_won%_-5,away_home_day_diff_-5,away_away_day_diff_-5
Southampton-Bournemouth-09/20/2019,0,0,1,609,21,10,8,1.8,1.8,1.7,...,11,0,0,0,53,20,13,60.6,-1,-1
Burnley-Norwich City-09/21/2019,0,0,0,663,21,12,6,1.4,1.4,0.9,...,8,0,0,1,37,7,15,31.8,-1,-1
Everton-Sheffield Utd-09/21/2019,0,0,0,512,18,13,11,1.4,1.4,1.1,...,11,0,0,0,53,20,13,60.6,-1,-1
Leicester City-Tottenham-09/21/2019,1,1,0,521,19,12,3,1.2,0.4,0.2,...,13,0,0,0,39,12,10,54.5,-1,-1
Manchester City-Watford-09/21/2019,0,0,0,492,16,16,16,1.5,1.5,1.4,...,14,0,0,0,56,18,14,56.3,-1,-1


# Target Processing

In [14]:
seasons=[
    '2019-20',
    '2020-21',
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
]
data_path='../data/premier_league/'
all_target_df={}
for season in seasons:
    season_path=f'{data_path}/{season}/'
    all_target_df[season]=pd.read_csv(f'{season_path}/all_target_df.csv')
    all_target_df[season]['date']=all_target_df[season]['date'].apply(lambda s: datetime.strptime(s, '%B-%d-%Y'))

In [15]:
all_target_df=pd.concat(all_target_df.values(), axis=0)

In [16]:
all_target_df.index=all_target_df['home']+'-'+all_target_df['away']+'-'+all_target_df['date'].apply(lambda x: datetime.strftime(x, "%m/%d/%Y"))
all_target_df=all_target_df.sort_values(['date','home','away'])
all_target_df=all_target_df.drop(columns=['home', 'away'])
all_target_df=all_target_df.drop(columns='date')

In [17]:
all_target_df

,home_goals,away_goals,home_corners,away_corners,home_cards,away_cards,home_shots,away_shots,home_sots,away_sots
Liverpool-Norwich City-08/09/2019,3,1,8,1,0,2,15,12,7,5
Bournemouth-Sheffield Utd-08/10/2019,1,1,2,3,2,1,13,8,3,3
Burnley-Southampton-08/10/2019,3,0,2,6,0,0,10,11,4,3
Crystal Palace-Everton-08/10/2019,0,0,6,1,2,3,6,10,2,2
Tottenham-Aston Villa-08/10/2019,3,1,12,0,1,0,31,7,6,4
...,...,...,...,...,...,...,...,...,...,...
Nott'ham Forest-Fulham-09/28/2024,0,1,6,4,2,4,11,13,1,1
Wolves-Liverpool-09/28/2024,1,2,1,8,2,3,8,9,3,5
Ipswich Town-Aston Villa-09/29/2024,2,2,9,0,4,1,15,7,5,3
Manchester Utd-Tottenham-09/29/2024,0,3,5,3,5,3,11,24,2,10


In [18]:
target_config={
    'home_goals': (0,4,1),
    'away_goals': (0,4,1),
    'home_corners': (5,10,1),
    'away_corners': (5,10,1),
    'home_cards': (0,5,1),
    'away_cards': (0,5,1),
    'home_shots': (5,20,2),
    'away_shots': (5,20,2),
    'home_sots': (0,10,1),
    'away_sots':(0,10,1),
}

In [19]:
target_dfs={}
for col, tar_range in target_config.items():
    target_dfs[col]=all_target_df[[col]]
    u, l, step=tar_range
    for i in range(u, l+1, step):
        target_dfs[col][f'{col}_over_{i}']=target_dfs[col][col]>i
    target_dfs[col]=target_dfs[col].drop(columns=col)


/var/folders/5j/cgxgswl52bv6qlx0pdt0f8080000gn/T/ipykernel_99361/1079590121.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_dfs[col][f'{col}_over_{i}']=target_dfs[col][col]>i
/var/folders/5j/cgxgswl52bv6qlx0pdt0f8080000gn/T/ipykernel_99361/1079590121.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_dfs[col][f'{col}_over_{i}']=target_dfs[col][col]>i
/var/folders/5j/cgxgswl52bv6qlx0pdt0f8080000gn/T/ipykernel_99361/1079590121.py:6: SettingWithCopyWarning: 
A value is trying to be set on a 

In [20]:
target_dfs['home_goals'].loc[historical_feature.index]

,home_goals_over_0,home_goals_over_1,home_goals_over_2,home_goals_over_3,home_goals_over_4
Southampton-Bournemouth-09/20/2019,True,False,False,False,False
Burnley-Norwich City-09/21/2019,True,True,False,False,False
Everton-Sheffield Utd-09/21/2019,False,False,False,False,False
Leicester City-Tottenham-09/21/2019,True,True,False,False,False
Manchester City-Watford-09/21/2019,True,True,True,True,True
...,...,...,...,...,...
Nott'ham Forest-Fulham-09/28/2024,False,False,False,False,False
Wolves-Liverpool-09/28/2024,True,False,False,False,False
Ipswich Town-Aston Villa-09/29/2024,True,True,False,False,False
Manchester Utd-Tottenham-09/29/2024,False,False,False,False,False


# Modeling

In [37]:
from xgboost import XGBClassifier
# read data

X_train, X_test, y_train, y_test = train_test_split(historical_feature, target_dfs['home_goals'].loc[historical_feature.index], test_size=.2)
# create model instance
bst = XGBClassifier(n_estimators=2000, max_depth=5, learning_rate=0.01)
# fit model
bst.fit(X_train, y_train)
# make predictions
preds = bst.predict(X_test)

In [39]:
from sklearn.metrics import accuracy_score, precision_score
accuracy_score(y_test, preds), precision_score(y_test, preds, average='micro')

(0.3271276595744681, 0.7121212121212122)